# CineCue: 1. Cleaning

We need reliable ratings and movie details before exploring recommendation ideas.
This notebook reads the official MovieLens 1M files, checks their quality, and saves
clean CSV files. Run `python scripts/download_data.py` from the repository root first.

The [official dataset](https://grouplens.org/datasets/movielens/1m/) uses `::` separators.
`movies.dat` uses Latin-1 text, so reading it as UTF-8 can lose accented title characters.
The [dataset README and terms](https://files.grouplens.org/datasets/movielens/ml-1m-README.txt)
describe the schema and whole-star 1–5 ratings. We do not use `users.dat` in Phase 1.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Work from either the repository root or its notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "requirements.txt").is_file(), "Open this notebook from the cloned repository."
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)

In [2]:
rating_columns = ["user_id", "movie_id", "rating", "timestamp"]
movie_columns = ["movie_id", "title", "genres"]
for filename in ["ratings.dat", "movies.dat"]:
    assert (RAW / filename).is_file(), f"Missing {filename}; run the download script or copy local files."

# Read strings first, so malformed numbers can be counted instead of silently changed.
for filename, columns in [("ratings.dat", rating_columns), ("movies.dat", movie_columns)]:
    with (RAW / filename).open(encoding="latin-1") as source:
        for line_number, line in enumerate(source, start=1):
            if len(line.rstrip("\r\n").split("::")) != len(columns):
                raise ValueError(f"{filename}, line {line_number}: expected {len(columns)} fields")
ratings_raw = pd.read_csv(RAW / "ratings.dat", sep="::", engine="python",
                          names=rating_columns, encoding="latin-1", dtype="string",
                          keep_default_na=False, on_bad_lines="error", index_col=False)
movies_raw = pd.read_csv(RAW / "movies.dat", sep="::", engine="python",
                         names=movie_columns, encoding="latin-1", dtype="string",
                         keep_default_na=False, on_bad_lines="error", index_col=False)

def profile(frame, name):
    print(f"{name}: {frame.shape[0]:,} rows, {frame.shape[1]} columns")
    display(pd.DataFrame({"dtype": frame.dtypes.astype(str),
                          "nulls": frame.isna().sum(),
                          "blank_strings": frame.apply(lambda col: col.str.strip().eq("").sum())}))
    print(f"Exact duplicate rows beyond the first: {frame.duplicated().sum():,}")

profile(ratings_raw, "Raw ratings")
profile(movies_raw, "Raw movies")
print("Raw repeated user/movie rows beyond the first:",
      int(ratings_raw.duplicated(["user_id", "movie_id"]).sum()))
print("Raw duplicate movie keys beyond the first:", int(movies_raw.duplicated("movie_id").sum()))

Raw ratings: 1,000,209 rows, 4 columns


,dtype,nulls,blank_strings
user_id,string,0,0
movie_id,string,0,0
rating,string,0,0
timestamp,string,0,0


Exact duplicate rows beyond the first: 0
Raw movies: 3,883 rows, 3 columns


,dtype,nulls,blank_strings
movie_id,string,0,0
title,string,0,0
genres,string,0,0


Exact duplicate rows beyond the first: 0


Raw repeated user/movie rows beyond the first: 0
Raw duplicate movie keys beyond the first: 0


## Cleaning decisions

Each check below reports a count, including zero. Invalid ratings are written to a
separate local file with their source row number; they are excluded from the clean data.
An invalid ID is missing, nonnumeric, nonpositive, fractional, or outside signed 64-bit range.
Ratings must be whole numbers from 1 to 5. Timestamps must be nonnegative whole Unix
seconds, parseable in UTC, and earlier than March 2003 because this version was released
in February 2003. These are dataset-specific domain checks, not statistical outlier rules.
We keep valid ratings of 1 and 5 and all valid high-activity users and movies.

We trim and collapse title whitespace but keep `title_original` unchanged. Genres are
trimmed, deduplicated, sorted, and checked against the 18 official genres. Missing or
invalid movie metadata stops the run for investigation instead of silently losing ratings.
Identical normalized movie records can collapse; conflicting records for one movie ID
also stop the run to avoid choosing arbitrary metadata or creating a many-to-many join.

In [3]:
ratings = ratings_raw.copy()
ratings["source_row"] = np.arange(1, len(ratings) + 1)
checks = {}
numeric = {col: pd.to_numeric(ratings[col], errors="coerce") for col in rating_columns}
valid = pd.Series(True, index=ratings.index)

for col in ["user_id", "movie_id"]:
    values = numeric[col]
    good = values.notna() & values.gt(0) & values.lt(2**63) & values.mod(1).eq(0)
    checks[f"invalid_{col}"] = int((~good).sum())
    valid &= good

values = numeric["rating"]
good = values.notna() & values.between(1, 5) & values.mod(1).eq(0)
checks["invalid_rating"] = int((~good).sum())
valid &= good

seconds = numeric["timestamp"]
# Bound before conversion to avoid overflow on corrupt input.
release_limit = int(pd.Timestamp("2003-03-01", tz="UTC").timestamp())
good_seconds = seconds.notna() & seconds.ge(0) & seconds.lt(release_limit) & seconds.mod(1).eq(0)
parsed_time = pd.to_datetime(seconds.where(good_seconds), unit="s", errors="coerce", utc=True)
good = good_seconds & parsed_time.notna()
checks["invalid_timestamp"] = int((~good).sum())
valid &= good

checks["invalid_rating_rows_total"] = int((~valid).sum())
ratings.loc[~valid].to_csv(PROCESSED / "invalid_ratings.csv", index=False)
ratings = ratings.loc[valid].copy()
for col in rating_columns:
    ratings[col] = numeric[col].loc[valid].astype("int64")
ratings["rated_at"] = parsed_time.loc[valid]

allowed_genres = {"Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
                  "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
                  "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"}
movies = movies_raw.copy()
movies["title_original"] = movies["title"]
movies["title"] = movies["title"].str.replace(r"\s+", " ", regex=True).str.strip()
genre_lists = movies["genres"].fillna("").map(lambda text: sorted(set(g.strip() for g in text.split("|"))))
movie_ids = pd.to_numeric(movies["movie_id"], errors="coerce")
good_ids = movie_ids.notna() & movie_ids.gt(0) & movie_ids.lt(2**63) & movie_ids.mod(1).eq(0)
checks["invalid_metadata_id"] = int((~good_ids).sum())
checks["empty_title"] = int((movies["title"].isna() | movies["title"].eq("")).sum())
checks["invalid_or_empty_genres"] = int((~genre_lists.map(lambda items: set(items).issubset(allowed_genres))).sum())
display(pd.Series(checks, name="affected_rows").to_frame())
assert checks["invalid_metadata_id"] == checks["empty_title"] == checks["invalid_or_empty_genres"] == 0,     "Review invalid movie metadata before joining."
movies["movie_id"] = movie_ids.astype("int64")
movies["genres"] = genre_lists.map("|".join)
checks["duplicate_metadata_rows_removed"] = int(movies.duplicated(["movie_id", "title", "genres"]).sum())
movies = movies.drop_duplicates(["movie_id", "title", "genres"], keep="first")
checks["conflicting_movie_keys"] = int(movies.loc[movies.duplicated("movie_id", keep=False), "movie_id"].nunique())
assert checks["conflicting_movie_keys"] == 0, "Conflicting metadata needs manual review."

,affected_rows
invalid_user_id,0
invalid_movie_id,0
invalid_rating,0
invalid_timestamp,0
invalid_rating_rows_total,0
invalid_metadata_id,0
empty_title,0
invalid_or_empty_genres,0


## Repeated interactions and conflicts

An exact duplicate repeats all four original fields; we keep the first source row.
The same user and movie at a different timestamp is a valid repeat event and stays,
even if the rating changes. For conflicting ratings at the *same* user/movie/timestamp,
we keep the first row in the source file and log the number of conflicts. This rule
never replaces an earlier event using a later event. Source order resolves ties only;
it is not evidence that the chosen rating is more accurate. All records involved in
a same-time conflict are saved locally for inspection.

In [4]:
checks["exact_rating_duplicates_removed"] = int(ratings.duplicated(rating_columns).sum())
ratings = ratings.drop_duplicates(rating_columns, keep="first")
event_key = ["user_id", "movie_id", "timestamp"]
conflict_mask = ratings.duplicated(event_key, keep=False)
conflicts = ratings.loc[conflict_mask]
checks["same_time_conflict_groups"] = int(conflicts.groupby(event_key).ngroups)
checks["same_time_conflict_rows"] = len(conflicts)
conflicts.to_csv(PROCESSED / "same_time_conflicts.csv", index=False)
checks["same_time_conflict_rows_removed"] = int(ratings.duplicated(event_key).sum())
ratings = ratings.drop_duplicates(event_key, keep="first")
pair_sizes = ratings.groupby(["user_id", "movie_id"]).size()
checks["valid_repeated_pairs_retained"] = int(pair_sizes.gt(1).sum())
checks["repeat_events_beyond_first_retained"] = int((pair_sizes - 1).sum())
print("Conflict inspection:", "No same-time conflicts found." if conflicts.empty else
      f"{len(conflicts):,} rows saved to same_time_conflicts.csv for inspection.")
display(pd.Series({key: value for key, value in checks.items()
                   if any(word in key for word in ["duplicate", "conflict", "repeat"] )},
                  name="count").to_frame())

# A validated left join detects missing metadata without losing interactions.
joined = ratings.merge(movies, on="movie_id", how="left", validate="many_to_one", indicator=True)
checks["ratings_without_metadata"] = int(joined["_merge"].ne("both").sum())
checks["unrated_catalog_movies"] = int((~movies["movie_id"].isin(ratings["movie_id"])).sum())
display(pd.Series({key: checks[key] for key in ["ratings_without_metadata", "unrated_catalog_movies"]},
                  name="count").to_frame())
assert len(joined) == len(ratings)
assert checks["ratings_without_metadata"] == 0, "Resolve missing metadata; do not drop these ratings silently."
assert movies["movie_id"].is_unique
assert len(ratings_raw) == (len(ratings) + checks["invalid_rating_rows_total"]
    + checks["exact_rating_duplicates_removed"] + checks["same_time_conflict_rows_removed"])

Conflict inspection: No same-time conflicts found.


,count
duplicate_metadata_rows_removed,0
conflicting_movie_keys,0
exact_rating_duplicates_removed,0
same_time_conflict_groups,0
same_time_conflict_rows,0
same_time_conflict_rows_removed,0
valid_repeated_pairs_retained,0
repeat_events_beyond_first_retained,0


,count
ratings_without_metadata,0
unrated_catalog_movies,177


In [5]:
ratings = ratings.sort_values(["timestamp", "source_row"], kind="stable").reset_index(drop=True)
ratings.insert(0, "interaction_id", np.arange(1, len(ratings) + 1))
movies = movies.sort_values("movie_id").reset_index(drop=True)
assert ratings["interaction_id"].is_unique
assert ratings["rating"].between(1, 5).all()
assert ratings["rated_at"].notna().all()
ratings.to_csv(PROCESSED / "ratings_clean.csv", index=False)
movies.to_csv(PROCESSED / "movies_clean.csv", index=False, encoding="utf-8")

summary = {"raw_ratings": len(ratings_raw), "raw_movies": len(movies_raw),
           "clean_ratings": len(ratings), "users": int(ratings.user_id.nunique()),
           "rated_movies": int(ratings.movie_id.nunique()), "catalog_movies": len(movies),
           "first_rating_utc": str(ratings.rated_at.min()),
           "last_rating_utc": str(ratings.rated_at.max()), "checks": checks}
(PROCESSED / "cleaning_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
display(ratings.dtypes.astype(str).rename("clean_rating_dtype").to_frame())
display(movies.dtypes.astype(str).rename("clean_movie_dtype").to_frame())
display(Markdown(f"**Result:** {len(ratings):,} clean ratings from {ratings.user_id.nunique():,} users, "
                 f"covering {ratings.movie_id.nunique():,} rated movies out of {len(movies):,} catalog movies. "
                 f"Removed {checks['invalid_rating_rows_total']:,} invalid rows, "
                 f"{checks['exact_rating_duplicates_removed']:,} exact duplicates, and "
                 f"{checks['same_time_conflict_rows_removed']:,} conflicting tie rows. "
                 "Saved ratings_clean.csv and movies_clean.csv as UTF-8 CSV files."))

,clean_rating_dtype
interaction_id,int64
user_id,int64
movie_id,int64
rating,int64
timestamp,int64
source_row,int64
rated_at,"datetime64[s, UTC]"


,clean_movie_dtype
movie_id,int64
title,string
genres,object
title_original,string


**Result:** 1,000,209 clean ratings from 6,040 users, covering 3,706 rated movies out of 3,883 catalog movies. Removed 0 invalid rows, 0 exact duplicates, and 0 conflicting tie rows. Saved ratings_clean.csv and movies_clean.csv as UTF-8 CSV files.

The clean tables keep the original integer target. CSV does not store dtypes, so later
notebooks explicitly parse UTC dates and cast small feature columns. The generated
data and inspection files stay out of Git. Notebook 2 uses these tables for exploration.